<a href="https://colab.research.google.com/github/frasercrichton/ai-dde-hackthon/blob/feature%2Fleiden-guidelines-doc/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install  chromadb
!pip install git+https://github.com/huggingface/transformers.git triton

import logging


# Remove existing handlers (prevents duplicate logs)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Set up logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-pu06zp5u
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-pu06zp5u
  Resolved https://github.com/huggingface/transformers.git to commit d1b92369ca193da49f9f7ecd01b08ece45c2c9aa
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [9]:
from transformers import AutoTokenizer, AutoModel
from langchain.text_splitter import RecursiveCharacterTextSplitter
import torch

class EmbeddingsProcessor:


    # embeddings_processor = EmbeddingsProcessor('sentence-transformers/all-MiniLM-L6-v2')

    def __init__(self, model_name):

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()


    def create_embeddings(self, text):

        inputs = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True
        )
        # logger.info(f'inputs: {inputs}')

        if torch.cuda.is_available():
            logger.info('cuda available')
            self.model.to('cuda')
            inputs = {k: v.to('cuda') for k, v in inputs.items()}
        else:
            logger.warning('cuda not available!')


        with torch.no_grad():
            outputs = self.model(**inputs)

        return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy().tolist()


In [22]:
import chromadb
logger = logging.getLogger(__name__)

class RAGDatabase:

    client = None

    def __init__(self, collection_name):
        logger.info('Initialising.')

        if RAGDatabase.client is None:
            RAGDatabase.client = chromadb.Client()
            self.client = RAGDatabase.client

        # logger.info('Resetting.')
        # self.client.reset()
        self.collection = self.client.get_or_create_collection(name=collection_name)

    def store_documents(self, documents: list):

        kwargs = {
            "documents": [doc.get('text') for doc in documents],
            "embeddings": [doc.get('embedding') for doc in documents],
            "ids": [doc.get('id') for doc in documents]
        }

        metadata = [doc.get('metadata', {}) for doc in documents if doc.get('metadata') is not None]
        if len(metadata) > 0:
            kwargs["metadatas"] = metadata

        try:
            self.collection.add(**kwargs)
        except Exception as e:
            logging.error(f"Error adding documents: {e}")

    def get_collection(self, collection_name):
        collection = self.client.get_collection(collection_name)
        return collection.get()

    def query_with_embeddings(self, embedding_query, metadata_query=None, n_results=3):

# n_results=3 to 5 is generally a safe default for most applications.
# What If My Top n_results Are Not Useful?
# 	•	Increase the embedding quality → Use a better embedding model or fine-tune one.
# 	•	Apply re-ranking → Use a second model to score and reorder the results.
# 	•	Filter results by metadata → If using ChromaDB with metadata, filter based on relevant categories (e.g., category="landmarks").
# 	•	Use hybrid retrieval → Combine keyword-based search with embeddings for better results.

        # n_results=3
        # high precision 1-3
  #       Works well when information may be spread across multiple short documents.
	# •	Example: Answering questions that require synthesizing different perspectives, like a summary of multiple research papers.

        # more context 3 to 5
  #       # high recall (broad retrieval for re-ranking) 5 to 10+
  #       Recommended when re-ranking or filtering is applied after retrieval.
	# •	Example: Open-domain Q&A systems where an LLM will decide the most relevant information after fetching multiple candidates.

        query = {
            'query_embeddings': embedding_query,
            'n_results': n_results
        }

        if metadata_query:
            query['where'] = metadata_query

        logger.info(query)


        results = self.collection.query(**query)

        logger.info(results)

        # Here’s the logic for this:
        #   •	Lower distances indicate higher similarity (the documents are more relevant).
        #   •	Higher distances indicate lower similarity (the documents are less relevant).
        threshold = 50.0
        distances = results['distances'][0]

        if min(distances) > threshold:
            print(f'No relevant documents found (Distances {distances}).')
            return [{
                'text': 'No relevant documnts found for this query',
                'id': 'unknown',
                'metadata': {}
            }]
        else:
            print(f'Proceed with RAG... {distances}')

        return [
            {
                'text': doc_text,
                'id': doc_id,
                'metadata': doc_metadata
            }
            for doc_text, doc_id, doc_metadata in zip(results['documents'][0], results['ids'][0], results['metadatas'][0])
        ]


    def query_with_text(self, query, filter, n_results=3):

        results = self.collection.query(
            query_texts=[query], where=filter, n_results=n_results
        )
        return [
            {
                'text': doc_text,
                'id': results['ids'][0][i],
                'metadata': results['metadatas'][0][i],
            }
            for i, doc_text in enumerate(results['documents'][0])
        ]

    def delete_collection(self, collection_name):
        self.client.delete_collection(collection_name)

    def reset_database(self):
        self.client.reset()

In [24]:
class MetadataQuery:

    def createFilters(self, headers: list, tags: list =None, context=None):

      filter_dict = {}

      if headers:
          if (len(headers) > 1):
            filter_dict['headers'] = {'$in': headers}
          else:
            filter_dict['headers'] = headers[0]

      if tags:
          filter_dict['tags'] = [{'$contains': tag} for tag in tags]

      if len(filter_dict.keys()) > 1:
          return {'$or': [{key: value} for key, value in filter_dict.items()]}
      else:
          return filter_dict

metadata_query = MetadataQuery().createFilters(headers=['VIDEOS'])
print(metadata_query)

metadata_query = MetadataQuery().createFilters(headers=['VIDEOS', 'PHOTOGRAPHS'])
print(metadata_query)

{'headers': 'VIDEOS'}
{'headers': {'$in': ['VIDEOS', 'PHOTOGRAPHS']}}


In [11]:
rag_database = RAGDatabase(collection_name='leiden_guidelines')
embeddings_processor = EmbeddingsProcessor('sentence-transformers/all-MiniLM-L6-v2')
# rag_database.delete_all(collection_name='my_collection')
documents =[
    {'text': 'The Eiffel Tower is located in Paris, France.'},
    {'text': 'The Great Wall of China is one of the Seven Wonders of the World.'},
    {'text': 'Python is a popular programming language for data science.'},
    {'text': 'Leonardo da Vinci painted the Mona Lisa.'},
]

for i, doc in enumerate(documents):
    doc['id'] = i


documents = [
    {
        **document,
        'id': str(i),
        'embedding': embeddings_processor.create_embeddings(document['text'])
    }
    for i, document in enumerate(documents)
]

print(documents)

rag_database.store_documents(documents)
query_embeddings = embeddings_processor.create_embeddings("who painted the Mona Lisa?")
x = rag_database.query_with_embeddings(query_embeddings)
logger.info(f'search result {x}')

INFO: Initialising.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
INFO: search result [{'text': 'Leonardo da Vinci painted the Mona Lisa.', 'id': '3', 'metadata': None}, {'text': 'The Great Wall of China is one of the Seven Wonders of the World.', 'id': '1', 'metadata': None}, {'text': 'The Eiffel Tower is located in Paris, France.', 'id': '0', 'metadata': None}]


[{'text': 'The Eiffel Tower is located in Paris, France.', 'id': '0', 'embedding': [0.25255128741264343, 0.261593759059906, 0.04478899762034416, -0.004811152815818787, 0.1809592992067337, -0.06470618396997452, -0.41695189476013184, 0.044289134442806244, 0.20888380706310272, -0.09429414570331573, 0.11149588972330093, -0.18263275921344757, 0.3690599501132965, -0.48395201563835144, 0.03566158190369606, -0.2918674051761627, -0.07082509249448776, -0.12295816838741302, 0.06394852697849274, -0.3695654571056366, 0.3303406834602356, -0.5041595101356506, 0.21871860325336456, -0.08313480764627457, -0.3599214255809784, -0.09521540254354477, -0.33874282240867615, 0.13332165777683258, -0.07765697687864304, -0.26415830850601196, 0.2931792438030243, -0.16103528439998627, -0.3515162765979767, 0.2917000353336334, -0.16635742783546448, 0.14924100041389465, 0.15074694156646729, -0.37358352541923523, 0.010461081750690937, -0.13055160641670227, 0.03111202083528042, 0.061918314546346664, -0.08030898123979568

In [12]:

documents = [
  {
  'metadata': {
   "headers": "VIDEOS",
    "tags": [
      "no excerpts",
      "entire video",
      "complete footage",
      "full recording"
    ]},
    "text": "Submission of videos in full, alongside their respective transcripts and translations, assist the Court in contextualising the segments of the video that have been identified as most relevant by the tendering party.30 The ICC Trial Chamber in Ntaganda admitted a full video broadcast instead of only the excerpts submitted by the Defence in order to provide context to the security situation portrayed by the video in its entirety.31 Excerpts. If, nevertheless, a party seeks to tender excerpts, the tendering party should also clearly indicate whether the full footage was available and who extracted the segments of the video.32 The opposing party may tender additional excerpts to assist the Court in contextualising the segments sought to be admitted.33 The ICC Trial Chamber in Ntaganda granted the Prosecution’s request to admit extensions of video excerpts that had been tendered by the Defence in order to illustrate the reason behind the presence of community leaders at an event depicted in the video excerpts"
  },
  {
  'metadata': {

   "headers": "VIDEOS",
    "tags": [
      "translation",
      "integral evidence",
      "associated materials",
      "combined evidence"
    ]},
    "text": "Transcript and translation documents are written records designed to faithfully reflect the contents of the video for better comprehension.35 Consequently, each document and the video are treated as parts of the same evidence. The formal submission of a video automatically includes recognising the formal submission of associated transcripts and translations which have been duly disclosed.36 Similarly, it would be inconsistent to impose restrictions on one part but not the others.37 The ICC Trial Chamber in Katanga and Ngudjolo Chui granted a request to apply the same protective measures to the transcript and translation of a video that the Prosecution had been authorised to apply to the video itself.38 To facilitate the presentation of the evidence in court, the tendering party should, as early as practicable, indicate the segments of the video, transcript, and translation which it intends to use.39 The parties should also consult and resolve any disagreements about the transcripts or translations.40 No transcript is necessary if the purpose of the video is to demonstrate ambient sound;41 the Defence in Mladić at the ICTY was not required to transcribe the ambient sound of a firefight in a video it tendered."
  },
  {
    'metadata': {

    "headers": "VIDEOS",
    "tags": [
      "working language",
      "court language",
      "submission deadline",
      "chamber access"
    ]},
    "text": "Translation. Pursuant to Regulation 39(1) of the Regulations of the Court, all documents and materials filed with the Registry shall be in a working language of the Court. If segments of the video are not in a working language of the Court, those segments must be translated into a working language of the Court before they can be deemed admissible.43 The Prosecution has not complied with its disclosure obligations under Rule 77 of the ICC Rules of Procedure and Evidence until the translations have been provided to the Defence.44 The translation requirement is based on the accused's right to be informed of the evidence upon which the Prosecution intends to rely, including the nature, cause and content of the charge.45 Moreover, the Chamber must be in a position to fully understand the evidence upon which the parties intend to rely.46 Accuracy of Translation. Videos must be of a sufficient sound quality to facilitate translation. In Mladić, the ICTY Trial Chamber relied upon a video’s English/French subtitles ‘in order not to get stuck’, although the sound quality in the Bosnian/Croatian/Serbian (BCS) original of the video was so poor that it risked inaccurate translation.47 The Defence, however, was permitted to rely upon this video so the proceedings could continue, but it was instructed to find a better BCS version of the video.48 In the absence of a coherent and intelligible version"
  },
  {
      'metadata': {

    "headers": "PHOTOGRAPHS",
    "tags": [
      "content inference",
      "visual analysis",
      "proof from image"
    ]},
    "text": "This Guideline is derived from the ICC’s treatment of video evidence, but it can reasonably be applied to photographs as well. Caution should be exercised when considering a photograph since differences in personal perception can cause difficulties in making a definite finding.93 The Court will rely on the photograph only to the extent that it can make such a definite finding.94 The ICC Trial Chamber in Lubanga found that a reliable distinction can be drawn between individuals of different ages, based solely on the individuals’ appearance.95 Pursuant to Rule 63(4) of the ICC Rules of Procedure and Evidence, there is no strict legal requirement that a photograph has to be corroborated by other evidence for the Court to be able to rely on it and establish a specific fact"
  },
  {
      'metadata': {

    "headers": "PHOTOGRAPHS",
    "tags": [
      "date",
      "location",
      "author",
      "source",
      "chain of custody",
      "events depicted"
    ]},
    "text": "Based on Article 69(4) of the Rome Statute and Rules 63 and 64 of the ICC Rules of Procedure and Evidence, regarding the Court’s authority to rule on the relevance, probative value and admissibility of any evidence, photographs should be accompanied by reliable information on their date, location and events depicted. If the Court does not receive such information, photographs’ relevance to issues in the case and probative value cannot be determined.97 The ICC Trial Chamber in Ntaganda noted that since six photographs brought by the Prosecution were not dated, their relevance and probative value surrounding issues in the case could not be determined.98 It added that when photographs are dated, the parties seeking admission should provide evidence from which the Court can conclude that the dates are correct and fall within the temporal scope of the charges.99 The ICC Trial Chamber in Ntaganda also noted that certain dated photographs could have some relevance, including photos dated ambiguously (such as ‘08/07 2003’, which could be interpreted as either the 8th of July or August 7th) or by a range (‘January-February 2003’), but in the absence of any further reliable information as to the date, location and events depicted in the photographs, it could not admit them into evidence due to lack of probative value."
  },
  {
      'metadata': {

    "headers": "PHOTOGRAPHS",
    "tags": [
      "photo corroboration",
      "testimony verification",
      "eyewitness",
      "photo validation"
    ]},
    "text": "Where photographic evidence is of poor quality or it is unclear who took them and/or how they were developed, consistent testimonies from credible witnesses who were at the site can corroborate the content of the photographs.103 The ICC Trial Chamber in Ntaganda noted the consistency of evidence from photographs taken from credible witnesses, and the consistent testimony from seven witnesses, with which it was able to satisfy itself that the photographs did indeed depict the aftermath of a massacre.104 Unreliable Expert Testimony. An expert witness’ testimony is unreliable if it is based on conclusions drawn from photographs displaying obvious limitations in terms of reliability.105 In Mladić, the ICTY Trial Chamber was presented with multiple photographs of the allegedly same crater: one was taken initially by a war correspondent during the conflict in the 1990s, and then others were subsequently taken by Defence experts in 2010. The Trial Chamber found the Defence expert’s conclusions drawn from the photographs were unreliable because of the limitations of the photographs in terms of their reliability.106 Firstly, the Chamber found that the photographs did not in fact depict the same crater, nor the same floor tiles which were depicted in the initial photograph.107 Secondly, editing software was used on the Defence expert’s photographs to place each photograph in a vertical position and remove deformations"
  },
  {
  'metadata': {

   "headers": "AERIAL AND SATELLITE IMAGES",
    "tags": [
      "expert report",
      "satellite image",
      "summary submission",
      "voluminous data"
    ]},
    "text": "Pursuant to Rule 92 bis (A) of the ICTY Rules of Procedure and Evidence, evidence of a witness in the form of a written statement may be admitted in lieu of oral testimony which goes to proof of a matter other than the acts and conduct of an accused as charged in the indictment. An example of the rule’s application would be if evidence in question is of a cumulative nature in that other witnesses will give, or have given, oral testimony of similar facts. This allows investigators to produce summary reports which are derived from multiple sources and aims to give background evidence to the forensic examinations, thereby contextualising and reducing the apparent complexity of their findings.117 ‘To facilitate matters and to speed up the process’,118 the ICTY in Krstić authorised an investigator with the Office of the Prosecutor to testify in a summary form about the findings of forensic experts who had conducted examinations of various grave sites in 1996, 1998 and 1999 ‘associated with the take-over of Srebrenica’."
  },
  {
      'metadata': {

    "headers": "AERIAL AND SATELLITE IMAGES",
    "tags": [
      "inseparable evidence",
      "satellite corroboration",
      "aerial photo",
      "linked testimony"
    ]},
    "text": "Pursuant to Rule 92 bis (D) of the ICTY Rules of Procedure and Evidence,120 ‘a Chamber may admit a transcript of evidence given by a witness in proceedings before the Tribunal which goes to proof of a matter other than the acts and conduct of the accused’. Although Rule 92 bis (D) does not explicitly provide for the admission of exhibits admitted during former testimony, these exhibits are admissible pursuant to this rule so long as they form an inseparable and indispensable part of the testimony (whether expert or not).121 Aerial and satellite images are an inseparable and indispensable part of the testimony if the witness discusses them ‘in his or her written statement or transcript and if that written statement would become incomprehensible or have lesser probative value without [the] admission’ of such images.122 Indexes. Aerial and satellite images admitted during former witness testimony should be tendered with an index. The index should indicate the exact title or exhibit number for each former exhibit to identify the exact exhibits from the previous case.123 The ICTY in Blagojević and Jokić postponed the admission of aerial images that had been previously tendered and admitted at the ICTY during related witness testimony of previous ICTY trials until an index ofproposed exhibits could be provided.1"
  }
]


In [13]:
import json
def format_documents_for_chroma(documents):
    formatted = []
    for doc in documents:
        metadata = doc.get('metadata', {})
        # Convert tags list to comma-separated string if present

        if 'tags' in metadata and isinstance(metadata['tags'], list):
            metadata['tags'] = ', '.join(metadata['tags'])
        formatted.append({
            'metadata': metadata,
            'text': doc['text']
        })
    return formatted

documents = format_documents_for_chroma(documents)

print(documents)

[{'metadata': {'headers': 'VIDEOS', 'tags': 'no excerpts, entire video, complete footage, full recording'}, 'text': 'Submission of videos in full, alongside their respective transcripts and translations, assist the Court in contextualising the segments of the video that have been identified as most relevant by the tendering party.30 The ICC Trial Chamber in Ntaganda admitted a full video broadcast instead of only the excerpts submitted by the Defence in order to provide context to the security situation portrayed by the video in its entirety.31 Excerpts. If, nevertheless, a party seeks to tender excerpts, the tendering party should also clearly indicate whether the full footage was available and who extracted the segments of the video.32 The opposing party may tender additional excerpts to assist the Court in contextualising the segments sought to be admitted.33 The ICC Trial Chamber in Ntaganda granted the Prosecution’s request to admit extensions of video excerpts that had been tende

In [30]:
embeddings_processor = EmbeddingsProcessor('sentence-transformers/all-MiniLM-L6-v2')

rag_database = RAGDatabase(collection_name='leiden_guidelines')

class DocumentParser:

  def parse_for_chroma(self, documents):
    for i, doc in enumerate(documents):
        doc['id'] = i

    return [
        {
            **document,
            'id': str(i),
            'embedding': embeddings_processor.create_embeddings(document['text'])
        }
        for i, document in enumerate(documents)
    ]


chroma_documents = DocumentParser().parse_for_chroma(documents)
print(chroma_documents)

rag_database.store_documents(chroma_documents)

query_embeddings = embeddings_processor.create_embeddings("who painted the Mona Lisa?")
search_results = rag_database.query_with_embeddings(query_embeddings)

import json
formatted_json = json.dumps(search_results, indent=4)
print(formatted_json)



INFO: Initialising.
INFO: {'query_embeddings': [-0.2785819470882416, -0.03609742224216461, 0.04675167798995972, 0.4694271683692932, -0.34219783544540405, 0.15185096859931946, 0.36336711049079895, -0.1785910278558731, -0.19310206174850464, -0.787968635559082, -0.06673038750886917, -0.42522311210632324, 0.24272549152374268, 0.049909044057130814, -0.39096078276634216, 0.38057494163513184, -0.0516190305352211, 0.31540513038635254, 0.3130388855934143, 0.15354330837726593, 0.0025088381953537464, -0.3067297637462616, -0.2909950017929077, 0.17102886736392975, 0.3957807421684265, 0.44371497631073, -0.08577834814786911, -0.6038872003555298, 0.2301788628101349, -0.07165664434432983, -0.53993159532547, -0.19256822764873505, -0.053473327308893204, -0.12437615543603897, -0.07120561599731445, 0.19195641577243805, 0.37520337104797363, 0.5540621876716614, 0.6651857495307922, -0.06441912800073624, -0.6954887509346008, -0.26026755571365356, -0.013121556490659714, -0.1416499763727188, 0.47964584827423096,

[{'metadata': {'headers': 'VIDEOS', 'tags': 'no excerpts, entire video, complete footage, full recording'}, 'text': 'Submission of videos in full, alongside their respective transcripts and translations, assist the Court in contextualising the segments of the video that have been identified as most relevant by the tendering party.30 The ICC Trial Chamber in Ntaganda admitted a full video broadcast instead of only the excerpts submitted by the Defence in order to provide context to the security situation portrayed by the video in its entirety.31 Excerpts. If, nevertheless, a party seeks to tender excerpts, the tendering party should also clearly indicate whether the full footage was available and who extracted the segments of the video.32 The opposing party may tender additional excerpts to assist the Court in contextualising the segments sought to be admitted.33 The ICC Trial Chamber in Ntaganda granted the Prosecution’s request to admit extensions of video excerpts that had been tende

In [32]:
query_embeddings = embeddings_processor.create_embeddings("tell me about formal submission of a video transcripts and traslation")
# query_embeddings = embeddings_processor.create_embeddings("Who is scooby doo?")


metadata_query = MetadataQuery().createFilters(headers=['VIDEOS'])

print(f'metadata_query: {metadata_query}')
rag_database = RAGDatabase(collection_name='leiden_guidelines')

search_results = rag_database.query_with_embeddings(query_embeddings, metadata_query)

import json
formatted_json = json.dumps(search_results, indent=4)
print(formatted_json)


INFO: Initialising.
INFO: {'query_embeddings': [-0.2700476348400116, -0.025295548141002655, -0.3370032012462616, -0.4905383586883545, 0.23973876237869263, 0.23945504426956177, 0.026041362434625626, 0.10438419878482819, 0.1445663869380951, -0.26186320185661316, 0.05859977751970291, -0.19734503328800201, -0.2452223002910614, 0.0021274781320244074, -0.40702420473098755, -0.40731683373451233, 0.015267584472894669, 0.1530831754207611, -0.1851809024810791, -0.27654212713241577, 0.5628379583358765, -0.007232723757624626, 0.13719314336776733, 0.2391466349363327, 0.13792960345745087, 0.09820380061864853, -0.21245472133159637, 0.1177147775888443, 0.41667789220809937, -0.07528974860906601, -0.2308485209941864, 0.032894961535930634, 0.5394834876060486, 0.2589340806007385, -0.026723885908722878, 0.13190633058547974, 0.08501525223255157, -0.18554821610450745, -0.28330934047698975, -0.30842065811157227, -0.1912803053855896, -0.296766072511673, 0.2932521402835846, -0.178111270070076, -0.13404394686222

metadata_query: {'headers': 'VIDEOS'}


ValueError: min() arg is an empty sequence

In [29]:
import pandas as pd

results = rag_database.get_collection('leiden_guidelines')

# Convert to DataFrame
df = pd.DataFrame({
    'id': results['ids'],
    'document': results['documents'],
    'metadata': results['metadatas'],
    'embedding': results['embeddings']  # optional: can be large
})

# Display
df.head(50)

,id,document,metadata,embedding
0,0,"The Eiffel Tower is located in Paris, France.",None,None
1,1,The Great Wall of China is one of the Seven Wo...,None,None
2,2,Python is a popular programming language for d...,None,None
3,3,Leonardo da Vinci painted the Mona Lisa.,None,None
4,4,Based on Article 69(4) of the Rome Statute and...,"{'tags': 'date, location, author, source, chai...",None
5,5,Where photographic evidence is of poor quality...,"{'tags': 'photo corroboration, testimony verif...",None
6,6,Pursuant to Rule 92 bis (A) of the ICTY Rules ...,"{'tags': 'expert report, satellite image, summ...",None
7,7,Pursuant to Rule 92 bis (D) of the ICTY Rules ...,"{'tags': 'inseparable evidence, satellite corr...",None


In [25]:

x = MetadataQuery().createFilters(headers='header text', tags=['tags', 'xxx'], context='context text')

import json
print(x)
formatted_json = json.dumps(x, indent=4)
print(formatted_json)

{'$or': [{'header': {'$in': 'header text'}}, {'tags': [{'$contains': 'tags'}, {'$contains': 'xxx'}]}]}
{
    "$or": [
        {
            "header": {
                "$in": "header text"
            }
        },
        {
            "tags": [
                {
                    "$contains": "tags"
                },
                {
                    "$contains": "xxx"
                }
            ]
        }
    ]
}
